# LATTE on Attar: timing-driven detailed placement in a few hundred lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayChang987/Attar/blob/main/examples/colab/latte_on_attar.ipynb)

This notebook runs **LATTE** (*Legality-Assured differentiable Timing-driven detailed placement*, DAC 2026) on
**[Attar](https://github.com/RayChang987/Attar)**, a GPU-tensor physical-design platform, on a contest design, in Colab.

Attar provides everything except the algorithm: design loading (the zoo), an OpenSTA-equivalent static timer,
global routing with routed parasitics, incremental legalization, legality checks, and a WL-driven detailed placer
for comparison. The LATTE code itself is 670 lines (`dsta.py`, the differentiable timer anchored on Attar's exact
timing, 373 lines; `latte.py`, the optimization loop, 297 lines).

**What runs here:**
1. Install Attar (pip) and write the LATTE sources.
2. Load ICCAD 2025 `des` (2.5k cells, ASAP7) from Attar's zoo; pre-CTS timing (ideal clock with the SDC's pin latencies).
3. Start from a DREAMPlace placement (as in the paper), measure timing after Attar's global routing.
4. Run LATTE for a few rounds, compare with Attar's WL-driven detailed placer, and plot.

**Runtime:** about 25 s per LATTE round on Colab's 2 CPU cores (GPU is optional; it helps more on larger designs).
The default 12 rounds take about 5 minutes and give most of the improvement.

## 1. Install Attar

In [ ]:
!pip install -q attar==1.0.4      # attar_patches.py below patches 1.0.4's internals
import torch, attar
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('attar', attar.__version__, '| torch', torch.__version__, '| device', DEVICE)

## 2. The LATTE sources

Three files, written to the working directory. `attar_patches.py` holds two fixes to Attar 1.0.4 found during this
work (pin clock latencies on ideal clocks; a router index on non-square grids); they are being upstreamed, after
which this file goes away.

In [ ]:
%%writefile attar_patches.py
"""Runtime fixes to Attar 1.0.4 used by this study (to be upstreamed to ~/attar; the shared checkout is not edited).

1. attar.route.router._Grid2D.path: both L-shape costs are evaluated for every branch and torch.where picks one; the
   vertical-first cost reads `mid` as a row index, but a horizontal-first branch's `mid` is a column. On a grid with
   more columns than rows (e.g. ICCAD 2025 des at a DREAMPlace placement) that index runs past the cost table and
   the CUDA gather asserts. Clamping `mid` to each formula's range does not change the selected cost.
"""
import torch
from attar.route import router as _router

_orig_path = _router._Grid2D.path


def _path(self, seg, typ, mid, over=False, via=0.0):
    ax, ay, bx, by = seg.unbind(1)
    mx = mid.clamp(0, self.nx - 1)              # mid as a column (horizontal-first)
    my = mid.clamp(0, self.ny - 1)              # mid as a row (vertical-first)
    c0 = self.h(ay, ax, mx, over) + self.v(mx, ay, by, over) + self.h(by, mx, bx, over)
    c1 = self.v(ax, ay, my, over) + self.h(my, ax, bx, over) + self.v(bx, my, by, over)
    if via:
        c0 = c0 + via * (ay != by) * ((mx != ax).float() + (mx != bx).float())
        c1 = c1 + via * (ax != bx) * ((my != ay).float() + (my != by).float())
    return torch.where(typ == 0, c0, c1)


_router._Grid2D.path = _path


# 2. Pin clock latencies on ideal clocks (set_clock_latency on register clock pins with set_propagated_clock removed,
#    i.e. pre-CTS timing of the ICCAD 2025 SDCs, which carry Innovus' useful-skew pin latencies). Attar 1.0.4 models
#    pin latencies on propagated clocks only and warns "pin clock latencies on ideal clocks are not modeled".
#    OpenSTA: the ideal clock network delay at such a pin is the pin's latency (instead of the clock's), at launch and
#    at capture. Here: launch arrivals (Timer.lc's t0) and capture latencies (_cap_lat) use it.
import warnings as _warnings
import numpy as _np
from attar.timing import sta as _sta

_orig_prepare_static = _sta.Timer._prepare_static
_orig_cap_lat = _sta.Timer._cap_lat


def _prepare_static(self):
    with _warnings.catch_warnings():
        _warnings.filterwarnings('ignore', message='.*pin clock latencies on ideal clocks.*')
        _orig_prepare_static(self)
    self._ideal_pin_lat = None
    pl = self.info.get('pin_latency') or []
    cidx = {k: i for i, k in enumerate(self.clock_names)}
    val = _np.full(self.P, _np.nan)
    for e in pl:
        p = int(e['pin'])
        ci = cidx.get(e.get('clock')) if e.get('clock') in cidx else (int(self.clk_of[p]) if self.clk_of[p] >= 0 else None)
        if ci is None or self.prop_clk[ci]:
            continue
        if sorted(e.get('rf', [0, 1])) != [0, 1] or sorted(e.get('mm', ['max', 'min'])) != ['max', 'min']:
            raise NotImplementedError('ideal-clock pin latency with -rise/-fall/-min/-max: %r' % (e,))
        val[p] = float(e['value'])
    if not _np.isfinite(val).any():
        return
    dev = self.dev
    pin_lat = torch.as_tensor(val, device=dev, dtype=torch.float32)
    self._ideal_pin_lat = pin_lat
    if self.lc is not None:
        lib, to, orf, tag, t0, cks = self.lc
        fr = self.l_fr[torch.as_tensor(self._lc_src, device=dev)]
        ck = torch.as_tensor(self.clk_of, device=dev)[fr]
        v = pin_lat[fr]
        t0 = torch.where(torch.isfinite(v), t0 - self.lat[ck] + v, t0)
        self.lc = (lib, to, orf, tag, t0, cks)


def _cap_lat(self, early):
    lat = _orig_cap_lat(self, early)
    pl = getattr(self, '_ideal_pin_lat', None)
    if pl is None or not len(self.k_lib):
        return lat
    v = pl[self.k_fr]
    return torch.where((torch.isfinite(v) & ~self.k_prop)[:, None], v[:, None], lat)


_sta.Timer._prepare_static = _prepare_static
_sta.Timer._cap_lat = _cap_lat

In [ ]:
%%writefile dsta.py
"""Differentiable STA for LATTE (paper §3.2), built on Attar's exact timer.

Forward: GBA max propagation of setup arrivals (rise / fall, per clock tag) level by level over Attar's timing graph,
cell delays from the NLDM tables (bilinear, differentiable), wire delays from the Elmore delay of an RC tree per net
(rooted rectilinear spanning tree over the pins, the driver at the root; pi segments), driver load = the tree's
capacitance + the sink pin capacitances, sink slew degraded by the wire (PERI: sqrt(s^2 + (ln 9 m1)^2)).

Anchoring ("delta" form): at the anchor placement pos0 (where the exact timer ran) every arc delay and slew equals the
exact timer's (OpenSTA's DMP delay calculation on estimate_parasitics trees); away from it each moves by the model's
change: d(pos) = d_exact(pos0) + d_model(pos) - d_model(pos0). So the forward is exact at pos0 and first-order
accurate nearby, and the gradient is the model's (Elmore + NLDM), as in the paper. Required times (setup checks,
output delays) and the clock network's arrivals are held at their pos0 values (the clock is not moved by DP).

TNS = sum over endpoints of min(slack, 0), WNS = min slack (subgradients through the maxima, the paper's GBA).
"""
import math

import numpy as np
import torch

from attar.timing.sta import nldm

INF = float('inf')
K_SLEW = math.log(9.0)
BUCKETS = (2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256, 384, 512, 768, 1024)


def _smooth_abs(x, eps=1e-3):
    return torch.sqrt(x * x + eps * eps)


class Trees:
    """Rooted RC trees of the data nets (fixed topology between rebuilds): per degree bucket the pins (driver first),
    each node's parent, the ancestor matrix (u on the path root -> v, u != root) and the child matrix."""

    def __init__(self, d, nets, drivers, pin_cap, pins_xy):
        dev = d.device
        self.dev = dev
        P = len(pin_cap)
        p_net = d.p_net.cpu().numpy()
        order = np.argsort(p_net, kind='stable')
        ptr = np.searchsorted(p_net[order], np.arange(len(d.n_ptr)))
        deg = ptr[1:] - ptr[:-1]
        self.buckets = []
        xy = pins_xy.detach()
        nets = np.asarray(nets)
        dn = deg[nets]
        sizes = sorted(set(b for b in BUCKETS if b < int(dn.max(initial=2))) | {max(int(dn.max(initial=2)), 2)})
        lo = 0
        for n in sizes:
            sel = nets[(dn > lo) & (dn <= n)]
            if len(sel):
                self.buckets.append(self._bucket(sel, n, ptr, order, drivers, xy, pin_cap, P))
            lo = n

    def _bucket(self, sel, n, ptr, order, drivers, xy, pin_cap, P):
        dev = self.dev
        B = len(sel)
        pins = np.full((B, n), P, np.int64)                      # P: a dummy pin (zero position, zero cap)
        cnt = ptr[sel + 1] - ptr[sel]
        row = np.repeat(np.arange(B), cnt)
        col = np.arange(int(cnt.sum())) - np.repeat(np.cumsum(cnt) - cnt, cnt)
        flat = order[np.repeat(ptr[sel], cnt) + col]
        pins[row, col] = flat
        # the driver to slot 0 (swap)
        dv = drivers[sel]
        where = np.nonzero(pins == dv[:, None])
        slot = np.zeros(B, np.int64); slot[where[0]] = where[1]
        first = pins[:, 0].copy()
        pins[np.arange(B), slot] = first
        pins[:, 0] = dv
        pins = torch.as_tensor(pins, device=dev)
        valid = pins < P
        X = torch.cat([xy, torch.zeros(1, 2, device=dev, dtype=xy.dtype)])[pins]          # (B, n, 2)
        with torch.no_grad():
            D = (X[:, :, None, :] - X[:, None, :, :]).abs().sum(-1)
            D = D.masked_fill(~valid[:, None, :] | ~valid[:, :, None], INF)
            intree = torch.zeros(B, n, dtype=torch.bool, device=dev); intree[:, 0] = True
            best = D[:, 0].clone(); par = torch.zeros(B, n, dtype=torch.long, device=dev)
            ins = torch.zeros(B, n, dtype=torch.long, device=dev)    # insertion order
            ar = torch.arange(B, device=dev)
            for k in range(1, n):
                cand = best.masked_fill(intree, INF)
                j = cand.argmin(1)
                ok = torch.isfinite(cand[ar, j])
                j = torch.where(ok, j, torch.zeros_like(j))
                ins[:, k] = j
                intree[ar[ok], j[ok]] = True
                dj = D[ar, j]
                better = (dj < best) & ~intree & ok[:, None]
                best = torch.where(better, dj, best)
                par = torch.where(better, j[:, None].expand(-1, n), par)
            anc = torch.zeros(B, n, n, device=dev, dtype=xy.dtype)
            for k in range(1, n):
                v = ins[:, k]
                live = v > 0
                row = anc[ar, par[ar, v]].clone()
                row[ar, v] = 1.0
                anc[ar[live], v[live]] = row[live]
            nonroot = valid.clone(); nonroot[:, 0] = False
            ch = torch.zeros(B, n, n, device=dev, dtype=xy.dtype)
            bi, wi = torch.nonzero(nonroot, as_tuple=True)
            ch[bi, par[bi, wi], wi] = 1.0
        cap = torch.cat([pin_cap, torch.zeros(1, device=dev, dtype=pin_cap.dtype)])[pins].clone()
        cap[:, 0] = 0.0                                          # the driver's own pin is not a load
        return {'nets': torch.as_tensor(sel, device=dev), 'pins': pins, 'par': par, 'anc': anc, 'ch': ch,
                'nonroot': nonroot.to(xy.dtype), 'cap': cap * valid}

    def eval(self, pins_xy, r, c, n_nets, P):
        """Elmore delay per pin (P,), total capacitance per net (E,), wirelength per net (E,), differentiable."""
        xy = torch.cat([pins_xy, torch.zeros(1, 2, device=pins_xy.device, dtype=pins_xy.dtype)])
        m1_all, c_all, l_all, pin_all, net_all = [], [], [], [], []
        for b in self.buckets:
            X = xy[b['pins']]
            Xp = torch.gather(X, 1, b['par'][:, :, None].expand(-1, -1, 2))
            L = _smooth_abs(X - Xp).sum(-1) * b['nonroot']                       # (B, n) edge into each node
            nodecap = b['cap'] + 0.5 * c * (L + torch.bmm(b['ch'], L[:, :, None])[:, :, 0])
            cdown = torch.bmm(b['anc'].transpose(1, 2), nodecap[:, :, None])[:, :, 0]
            m1 = torch.bmm(b['anc'], (r * L * cdown)[:, :, None])[:, :, 0]
            m1_all.append(m1.flatten()); pin_all.append(b['pins'].flatten())
            c_all.append(nodecap.sum(1)); l_all.append(L.sum(1)); net_all.append(b['nets'])
        m1p = torch.zeros(P + 1, device=xy.device, dtype=xy.dtype).index_add(0, torch.cat(pin_all), torch.cat(m1_all))[:P]
        nets = torch.cat(net_all)
        cn = torch.zeros(n_nets, device=xy.device, dtype=xy.dtype).index_add(0, nets, torch.cat(c_all))
        ln = torch.zeros(n_nets, device=xy.device, dtype=xy.dtype).index_add(0, nets, torch.cat(l_all))
        return m1p, cn, ln


class DSTA:
    """Differentiable setup timing of design d, anchored at the current placement (exact timer run there)."""

    def __init__(self, d, max_degree=1024, ruler='gr', dtype=torch.float32):
        """ruler: the exact timing the DSTA is anchored on: 'gr' (Attar global routes + GR parasitics, as an early
        global route) or 'placement' (estimate_parasitics -placement)."""
        self.d = d
        self.max_degree = max_degree
        self.ruler = ruler
        self.dt = dtype
        self.anchor()

    def exact(self, pos=None):
        """Exact timing (TimingResult, timer) on the DSTA's ruler at pos (default: the design's placement)."""
        d = self.d
        if self.ruler == 'gr':
            routes, st = d.global_route(pos=pos, verbose=False)
            res = d.timing(pos=pos, routes=routes)
            return res, d._gr_timer[1], st
        res = d.timing(pos=pos)
        return res, d._timer, None

    # -- anchor: exact timing at pos0, trees, baselines ------------------------------------------------------------
    def anchor(self, pos=None):
        d = self.d
        if pos is not None:
            d.pos = pos
        res, T, self.gr_stats = self.exact()
        # clocked latches (mempool_tile_wrap: 5650): the exact timer passes late D data through D -> Q (time borrowing,
        # pseudo rows in T._cb_fwd); the model uses the levels' own arrival masks (cb[7]), which exclude those arcs, so
        # Q's arrival comes from the enable alone. At pos0 the endpoint offsets (ep_off) keep the slacks exact; the
        # gradient does not reach cells upstream of a borrowing latch through it.
        self.n_latch = int(getattr(T, '_lq_n', 0))
        self.T, self.res = T, res
        dev = d.device
        dt = self.dt
        self.dev = dev
        P, NT = T.P, T.NT
        self.P, self.NT = P, NT
        self.pos0 = d.pos.detach().clone()
        self.A0 = T.amax.reshape(P * 2, NT).detach().clone().to(dt)
        self.S0 = T.smax.reshape(-1).detach().clone().to(dt)
        self.C0 = T.load.detach().clone().to(dt)                    # (P, 2) total load per driver pin, rise / fall
        self.pin_cap = T.pin_cap.to(dt).mean(1)
        self.r, self.c = float(T.r_sig), float(T.c_sig)
        clk = T.clk_of >= 0
        drv = T.net_driver
        p_net = d.p_net.cpu().numpy()
        deg = np.bincount(p_net, minlength=len(drv))
        nets = np.nonzero((drv >= 0) & (deg >= 2) & (deg <= self.max_degree))[0]
        nets = nets[~clk[drv[nets]]]                              # the clock network stays as timed (DP keeps it)
        self.model_nets = nets
        self.p_net_t = d.p_net
        self.trees = Trees(d, nets, drv, self.pin_cap, self._pins(self.pos0.to(dt)))
        with torch.no_grad():
            self.m1_0, self.cn_0, _ = self.trees.eval(self._pins(self.pos0.to(dt)), self.r, self.c, len(drv), P)
        # levels: the ideal clock's launch arcs first (T.lc), then the levels of the search
        recs = list(T._rec)
        stages = []
        if T.lc is not None:
            lib, to, orf, tag, t0, cks = T.lc
            ea, en = T.lc_ex
            stages.append((dict(lib=lib, f=None, t_=to, orf=orf, irf=None, ea=ea, en=en,
                                arr=torch.ones(len(lib), dtype=torch.bool, device=dev), tag=tag, t0=t0, cks=cks),
                           recs.pop(0)))
        for cb in T.cb_lev:
            if cb is not None:
                lib, f, t_, orf, irf, ea, en, arr = cb
                stages.append((dict(lib=lib, f=f, t_=t_, orf=orf, irf=irf, ea=ea, en=en, arr=arr), recs.pop(0)))
        assert not recs
        self.levels = []
        for c, rec in stages:
            lib, t_, orf, ea, en, arr = c['lib'], c['t_'], c['orf'], c['ea'], c['en'], c['arr']
            launch = c['f'] is None
            fi = None if launch else c['f'] * 2 + c['irf']
            ti = t_ * 2 + orf
            tu, g_inv = torch.unique(ti, return_inverse=True)
            sk = T.n_to[en]
            si = sk * 2 + orf[ea]
            su, l_inv = torch.unique(si, return_inverse=True)
            C0 = self.C0[t_, orf]
            s_in0 = c['cks'].to(dt) if launch else self.S0[fi]
            dm0 = self._lookup(lib, orf, s_in0, C0)
            sm0 = self._lookup(lib, 2 + orf, s_in0, C0)
            smod0 = torch.full((len(tu),), -INF, device=dev, dtype=dt).scatter_reduce(0, g_inv, sm0, 'amax')
            tab_d = self._tables(lib, orf)
            tab_s = self._tables(lib, 2 + orf)
            L = dict(tab_d=tab_d, tab_s=tab_s, lib=lib, fi=fi, ti=ti, t=t_, orf=orf, tu=tu, g_inv=g_inv, ea=ea, sk=sk, su=su,
                     l_inv=l_inv, arr=arr, dg0=rec[0].detach().clone().to(dt),
                     wx0=rec[1].detach().clone().to(dt), dm0=dm0, sm0=sm0, smod0=smod0, C0=C0,
                     net_t=self.p_net_t[t_], launch=launch)
            if launch:
                L.update(tag=c['tag'], t0=c['t0'].to(dt), s_in=s_in0)
            self.levels.append(L)
        # nets driven by input ports: the sinks move with the wire only
        pa = T.pa
        self.pa_rows = torch.cat([T.n_to[pa] * 2, T.n_to[pa] * 2 + 1])
        self.pa_sink = torch.cat([T.n_to[pa], T.n_to[pa]])
        # endpoints: required times (pos0), CRPR and other credits as a constant offset per pin
        rows, reqs = [], []
        for rws, req, _ in T._setup_required():
            rows.append(rws); reqs.append(torch.where(torch.isnan(req), torch.full_like(req, INF), req).to(self.dt))
        self.ep_rows = torch.cat(rows); self.ep_req = torch.cat(reqs)
        self.ep_pin = self.ep_rows // 2
        self.ep_u, self.ep_inv = torch.unique(self.ep_pin, return_inverse=True)
        mine0 = self._slack_pins(self.A0)
        exact = res.setup.to(self.dt)[self.ep_u]
        fin = torch.isfinite(exact) & torch.isfinite(mine0)
        self.ep_ok = fin
        self.ep_idx = torch.nonzero(fin).flatten()
        self.ep_off = torch.where(fin, exact - mine0, torch.zeros_like(exact))
        self.exact_tns, self.exact_wns = res.tns, res.wns
        return self

    def _pins(self, pos):
        """Pin locations at pos with the orientations of the last sync_orient() (a buffer the DSTA owns: a CUDA
        graph keeps reading it; d.update_orient() replaces d.pin_off with a new tensor)."""
        if not hasattr(self, '_pin_off') or self._pin_off.dtype != pos.dtype:
            self._pin_off = self.d.pin_off.to(pos.dtype).clone()
            self._half = (self.d.size[self.d.p_cell] / 2).to(pos.dtype)
        elif not (pos.is_cuda and torch.cuda.is_current_stream_capturing()):
            self.sync_orient()
        return pos[self.d.p_cell] + self._half + self._pin_off

    def sync_orient(self):
        if hasattr(self, '_pin_off'):
            self._pin_off.copy_(self.d.pin_off)

    def _tables(self, lib, slot):
        T = self.T
        return (T.t_ix1[lib, slot].to(self.dt).contiguous(), T.t_ix2[lib, slot].to(self.dt).contiguous(),
                T.t_val[lib, slot].to(self.dt).contiguous(), T.t_n[lib, slot].contiguous())

    def _lookup(self, lib, slot, x, y):
        T = self.T
        return nldm(T.t_ix1[lib, slot].to(self.dt), T.t_ix2[lib, slot].to(self.dt), T.t_val[lib, slot].to(self.dt),
                    T.t_n[lib, slot], x, y)

    def _slack_pins(self, A):
        s = (self.ep_req - A[self.ep_rows]).amin(1)
        s = torch.where(torch.isfinite(s), s, torch.full_like(s, INF))
        return torch.full((len(self.ep_u),), INF, device=self.dev, dtype=s.dtype).scatter_reduce(
            0, self.ep_inv, s, 'amin')

    # -- forward -----------------------------------------------------------------------------------------------------
    def slacks(self, pos):
        """Endpoint setup slacks (n,) at pos (differentiable), for the endpoints in self.ep_u[self.ep_ok]."""
        pos = pos.to(self.dt)
        pins = self._pins(pos)
        m1, cn, _ = self.trees.eval(pins, self.r, self.c, len(self.T.net_driver), self.P)
        dm1 = m1 - self.m1_0
        dcn = cn - self.cn_0
        A, S = self.A0, self.S0
        # input-port nets
        rows = self.pa_rows
        A = A.index_copy(0, rows, self.A0[rows] + dm1[self.pa_sink][:, None])
        for L in self.levels:
            fi = L['fi']
            s_in = L['s_in'] if L['launch'] else S[fi]
            C = L['C0'] + dcn[L['net_t']]
            dg = L['dg0'] + nldm(*L['tab_d'], s_in, C) - L['dm0']
            sm = nldm(*L['tab_s'], s_in, C)
            arr = L['arr'][:, None]
            NT = A.shape[1]
            if L['launch']:
                base = torch.full((len(dg), NT), -INF, device=self.dev, dtype=A.dtype)
                cand = base.scatter(1, L['tag'][:, None], (L['t0'] + dg)[:, None])
            else:
                cand = torch.where(arr, A[fi] + dg[:, None], torch.full_like(A[fi], -INF))
            An = torch.full((len(L['tu']), NT), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                0, L['g_inv'][:, None].expand(-1, NT), cand, 'amax')
            A = A.index_copy(0, L['tu'], An)
            smod = torch.full((len(L['tu']),), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                0, L['g_inv'], sm, 'amax')
            S = S.index_copy(0, L['tu'], self.S0[L['tu']] + smod - L['smod0'])
            if len(L['ea']):
                ea, sk = L['ea'], L['sk']
                w = L['wx0'] + dm1[sk]
                cl = torch.where(arr[ea], cand[ea] + w[:, None], torch.full_like(cand[ea], -INF))
                Ln = torch.full((len(L['su']), NT), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                    0, L['l_inv'][:, None].expand(-1, NT), cl, 'amax')
                A = A.index_copy(0, L['su'], Ln)
                sd = S[L['ti'][ea]]; sd0 = self.S0[L['ti'][ea]]
                dsl = torch.sqrt(sd * sd + (K_SLEW * m1[sk]) ** 2) - torch.sqrt(sd0 * sd0 + (K_SLEW * self.m1_0[sk]) ** 2)
                dsu = torch.full((len(L['su']),), -INF, device=self.dev, dtype=A.dtype).scatter_reduce(
                    0, L['l_inv'], dsl, 'amax')
                S = S.index_copy(0, L['su'], self.S0[L['su']] + dsu)
        s = self._slack_pins(A) + self.ep_off
        return s.index_select(0, self.ep_idx)

    def forward(self, pos):
        s = self.slacks(pos)
        tns = s.clamp(max=0).sum()
        wns = s.min().clamp(max=0)
        return tns, wns

    def wirelength(self, pos):
        """Tree wirelength of the modeled nets (um), differentiable."""
        _, _, ln = self.trees.eval(self._pins(pos.to(self.dt)), self.r, self.c, len(self.T.net_driver), self.P)
        return ln.sum()

    def cell_grad(self, pos=None):
        """G_i = dTNS / d(x_i, y_i) at pos (default: the anchor), (C, 2)."""
        pos = (self.pos0 if pos is None else pos).detach().to(self.dt).requires_grad_()
        tns, _ = self.forward(pos)
        g, = torch.autograd.grad(tns, pos)
        return g, float(tns)


class GraphedObjective:
    """-TNS - lam1 WNS and its gradient over all cell positions, as one CUDA graph replay (the DSTA's shapes are fixed
    between anchors). call(pos) -> (tns, wns, grad (C, 2)); the returned tensors are the graph's static outputs
    (valid until the next call)."""

    def __init__(self, ds, lam1):
        self.ds = ds
        dt = ds.dt
        self.static = ds.pos0.to(dt).clone().requires_grad_()
        self.graph = None
        if ds.dev.type != 'cuda':
            self.lam1 = lam1
            return
        st = torch.cuda.Stream(device=ds.dev)
        st.wait_stream(torch.cuda.current_stream(ds.dev))
        with torch.cuda.stream(st):
            for _ in range(2):
                tns, wns = ds.forward(self.static)
                g, = torch.autograd.grad(-tns - lam1 * wns, self.static)
        torch.cuda.current_stream(ds.dev).wait_stream(st)
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.tns, self.wns = ds.forward(self.static)
            self.g, = torch.autograd.grad(-self.tns - lam1 * self.wns, self.static)

    def __call__(self, pos):
        if self.graph is None:
            p = pos.detach().to(self.ds.dt).requires_grad_()
            tns, wns = self.ds.forward(p)
            g, = torch.autograd.grad(-tns - self.lam1 * wns, p)
            return tns.detach(), wns.detach(), g
        with torch.no_grad():
            self.static.copy_(pos)
            self.ds.sync_orient()
        self.graph.replay()
        return self.tns, self.wns, self.g

In [ ]:
%%writefile latte.py
"""LATTE: legality-assured differentiable timing-driven detailed placement (paper §3), on Attar.

Per round: anchor the DSTA (exact timing at the current legal placement) -> G = dTNS/d(x, y) -> top-K cells by |G|
(§3.3) -> balanced independent sets (Algorithm 1, §3.4) -> per set, in order of criticality: Adam with CAWP
preconditioning and warmup + cosine learning rate on F = -TNS - l1 WNS + l2 WL, the other cells fixed (§3.5),
then incremental legalization of the set (§3.6) -> exact timing; the round is reverted if TNS got worse.
"""
import json, math, os, sys, time
import numpy as np
import torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from attar import naive, Evaluator
from attar.io.distill import ASIDE
from dsta import DSTA, GraphedObjective

DEFAULTS = dict(
    rounds=30,              # maximum rounds N
    patience=3,             # stop after this many rounds without improvement
    k_frac=0.02,            # critical cells per round, fraction of the movable cells
    k_min=20,
    fanout_max=64,          # nets above this fanout do not make two cells adjacent (Algorithm 1)
    balance_T=1000,         # balancing iterations T (Algorithm 1)
    iters=40,               # Adam iterations per set
    lr=0.15,                # peak learning rate (um per step, Adam)
    warmup=5,               # warmup iterations, then cosine annealing to 0
    tau=1.0, eps=1e-12, T0=10,   # CAWP (Eq. 4)
    lam1=1.0,               # WNS weight (Eq. 3)
    lam2=0.3,               # WL weight, relative to |grad TNS| / |grad WL| at the set's first iteration
    radius=6.0,             # max move of a cell from its round-start position (um)
    max_sets=64,            # sets beyond this are merged into the last ones (runtime bound)
    cawp=True,
    select='grad',          # 'grad' (paper) or 'slack' (ablation C2)
    time_limit=7200.0,
    ruler='gr',             # exact timing the DSTA is anchored on
    filter='gr',            # bad-move filter: 'gr', 'placement', or 'both' (GR TNS and placement TNS improve)
    ignore_later=False,     # paper §3.6: legalize a set with the subsequent sets ignored (worse here, see notes)
    push_neighbors=False,   # variant: also re-legalize the non-critical cells a moved set overlaps
    graph=True,             # CUDA-graph the DSTA objective (per round)
    tabu_rounds=-1,         # cells of a rejected round sit out: -1 until a round is accepted (a retry on the
                            # unchanged placement repeats the same result), n > 0 for n rounds, 0 off
    revert_lr=1.0,          # learning-rate factor after a rejected round
    diag=True,              # DSTA gain before / after legalization per round (2 extra forwards per set)
)


def adjacency_nets(d, fanout_max, T):
    """Per cell, the nets that make it adjacent to another (no clock nets, fanout <= fanout_max)."""
    p_net = d.p_net
    deg = torch.bincount(p_net, minlength=len(d.n_ptr) - 1)
    drv = torch.as_tensor(T.net_driver, device=d.device)
    clk = torch.as_tensor(T.clk_of >= 0, device=d.device)
    clk_net = torch.zeros_like(deg, dtype=torch.bool)
    ok = drv >= 0
    clk_net[ok] = clk[drv[ok]]
    keep = (deg >= 2) & (deg <= fanout_max + 1) & ~clk_net
    return keep


def independent_sets(cells, score, d, keep_net, T_bal):
    """Algorithm 1: greedy coloring by score (descending), then balancing. cells, score: numpy."""
    p_cell = d.p_cell.cpu().numpy(); p_net = d.p_net.cpu().numpy(); keep = keep_net.cpu().numpy()
    cs = set(cells.tolist())
    nets_of = {c: set() for c in cells.tolist()}
    m = np.isin(p_cell, cells) & keep[p_net]
    for c, n in zip(p_cell[m].tolist(), p_net[m].tolist()):
        nets_of[c].add(n)
    order = cells[np.argsort(-score, kind='stable')]
    sc = dict(zip(cells.tolist(), score.tolist()))
    sets, used = [], []
    for c in order.tolist():
        for k in range(len(sets)):
            if not (nets_of[c] & used[k]):
                sets[k].append(c); used[k] |= nets_of[c]
                break
        else:
            sets.append([c]); used.append(set(nets_of[c]))
    # balancing: move the lowest-score movable cell of the largest set into the smallest one
    for _ in range(T_bal):
        sizes = [len(s) for s in sets]
        i, j = int(np.argmax(sizes)), int(np.argmin(sizes))
        if sizes[i] - sizes[j] <= 1:
            break
        moved = False
        for c in sorted(sets[i], key=lambda c: sc[c]):
            if not (nets_of[c] & used[j]):
                sets[i].remove(c); used[i] = set().union(*[nets_of[x] for x in sets[i]]) if sets[i] else set()
                sets[j].append(c); used[j] |= nets_of[c]
                moved = True
                break
        if not moved:
            break
    sets = [s for s in sets if s]
    sets.sort(key=lambda s: -max(sc[c] for c in s))
    return sets


def warmup_cosine(t, n, warm, lr):
    if t < warm:
        return lr * (t + 1) / warm
    return lr * 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, n - warm)))


def optimize_set(ds, d, S, cfg, base_pos, log, obj=None):
    """Adam + CAWP on the cells S (index tensor); other cells fixed at base_pos. Returns the new positions of S.
    obj: a dsta.GraphedObjective (-TNS - lam1 WNS and its gradient over all cells), else computed eagerly."""
    xy0 = base_pos[S].to(ds.dt)
    xy = xy0.clone().requires_grad_()
    opt = torch.optim.Adam([xy], lr=cfg['lr'])
    hist = []
    lam2 = None
    pfull = base_pos.to(ds.dt).clone()
    for t in range(cfg['iters']):
        for g_ in opt.param_groups:
            g_['lr'] = warmup_cosine(t, cfg['iters'], cfg['warmup'], cfg['lr'])
        with torch.no_grad():
            pfull[S] = xy
        if obj is not None:
            tns, wns, gall = obj(pfull)
            g_t = gall[S].clone()
        else:
            p = base_pos.to(xy.dtype).index_put((S,), xy)
            tns, wns = ds.forward(p)
            g_t, = torch.autograd.grad(-tns - cfg["lam1"] * wns, xy)
        pw = pfull.detach().float().requires_grad_()
        wl = d.wa_wirelength(pw, gamma=0.5)
        g_w = torch.autograd.grad(wl, pw)[0][S].to(ds.dt)
        if lam2 is None:
            nt, nw = float(g_t.norm()), float(g_w.norm())
            lam2 = cfg['lam2'] * (nt / nw if nw > 0 and nt > 0 else 0.0)
        g = (g_t + lam2 * g_w) * cfg.get('gunit', 1.0)    # gunit: objective-unit scale (CAWP mechanism check)
        if cfg['cawp']:
            if len(hist) < cfg['T0']:
                hist.append((g * g).sum(1))
            h = torch.stack(hist).mean(0)
            pre = cfg['tau'] / torch.sqrt(h + cfg['eps'])
            g = g * pre[:, None]
        xy.grad = g
        opt.step()
        with torch.no_grad():
            dlt = (xy - xy0).clamp(-cfg['radius'], cfg['radius'])
            xy.copy_(xy0 + dlt)
        if t == 0 or t == cfg['iters'] - 1:
            log.append({'t': t, 'tns': float(tns), 'wns': float(wns), 'wl': float(wl)})
    return xy.detach()



def run(d, cfg=None, log_path=None, verbose=True):
    cfg = dict(DEFAULTS, **(cfg or {}))
    t_start = time.time()
    ev = Evaluator(d)
    mv = torch.nonzero(d.movable).flatten()
    K = max(cfg['k_min'], int(cfg['k_frac'] * len(mv)))
    # core box for clamping
    rows_y = d.pos[mv, 1]
    xmin = float(d.pos[mv, 0].min()); xmax = float((d.pos[mv, 0] + d.size[mv, 0]).max())
    ymin = float(rows_y.min()); ymax = float((rows_y + d.size[mv, 1]).max())
    def exact(ruler):
        if ruler == 'gr':
            routes, _ = d.global_route(verbose=False)
            return d.timing(routes=routes).summary()
        if ruler == 'both':                    # GR timing, plus placement-parasitic TNS (router-independent)
            routes, _ = d.global_route(verbose=False)
            r = dict(d.timing(routes=routes).summary())
            p = d.timing().summary()
            r['p_tns'], r['p_wns'] = p['tns'], p['wns']
            return r
        return d.timing().summary()
    best = exact(cfg['filter'])
    hist = [{'round': 0, 'tns': best['tns'], 'wns': best['wns'], 'hpwl': float(d.hpwl()), 'accepted': True, 't': 0.0}]
    if verbose:
        print('start  TNS %.1f WNS %.2f HPWL %.1f' % (best['tns'], best['wns'], hist[0]['hpwl']), flush=True)
    keep_net = None
    bad = 0
    tabu, tabu_of = set(), {}             # cells of rejected rounds, kept out of the selection for tabu_rounds
    lr_scale = 1.0
    for r in range(1, cfg['rounds'] + 1):
        if time.time() - t_start > cfg['time_limit']:
            break
        tr = time.time()
        ds = DSTA(d, ruler=cfg['ruler'])
        if keep_net is None:
            keep_net = adjacency_nets(d, cfg['fanout_max'], ds.T)
        pos_r = d.pos.clone(); ori_r = d.orient.clone()
        g, tns_d = ds.cell_grad()
        if cfg['select'] == 'grad':
            score = g.norm(dim=1)
        else:                                        # ablation: worst slack of the cell's pins
            ps = ds.res.pin_slack()
            sc = torch.full((d.pos.shape[0],), float('inf'), device=d.device)
            sc = sc.scatter_reduce(0, d.p_cell, ps.float(), 'amin')
            score = torch.where(torch.isfinite(sc) & (sc < 0), -sc, torch.zeros_like(sc))
        score[~d.movable] = 0
        if tabu:
            score[torch.as_tensor(sorted(tabu), device=d.device)] = 0
        nz = int((score > 0).sum())
        k = min(K, nz)
        if k == 0:
            break
        crit = torch.topk(score, k).indices
        sets = independent_sets(crit.cpu().numpy(), score[crit].cpu().numpy(), d, keep_net, cfg['balance_T'])
        if len(sets) > cfg['max_sets']:
            extra = sets[cfg['max_sets']:]
            sets = sets[:cfg['max_sets']]          # (the least critical cells wait for a later round)
        base = d.pos.clone()
        c2 = dict(cfg); c2['lr'] = cfg['lr'] * lr_scale
        logs = []
        diag = {'pre': 0.0, 'post': 0.0, 'move_um': 0.0, 'lg_um': 0.0}
        set_t = [torch.as_tensor(S_, device=d.device) for S_ in sets]
        obj = None
        if cfg['graph']:
            try:
                obj = GraphedObjective(ds, cfg['lam1'])
            except RuntimeError as e:              # e.g. CUDA OOM while capturing (shared GPU): eager objective
                if 'out of memory' not in str(e):
                    raise
                obj = None
                torch.cuda.empty_cache()
                cfg['graph'] = False
                print('  (CUDA graph capture out of memory: eager DSTA objective from here on)', flush=True)
        for i_s, S in enumerate(set_t):
            xy = optimize_set(ds, d, S, c2, base, logs, obj)
            xy[:, 0] = torch.minimum(xy[:, 0].clamp(min=xmin), xmax - d.size[S, 0].to(xy.dtype))
            xy[:, 1] = torch.minimum(xy[:, 1].clamp(min=ymin), ymax - d.size[S, 1].to(xy.dtype))
            if cfg['diag']:
              with torch.no_grad():
                t_b = float(ds.forward(base)[0])
                pre = base.clone(); pre[S] = xy.to(pre.dtype)
                t_pre = float(ds.forward(pre)[0])
            p = d.pos.clone(); p[S] = xy.float(); d.pos = p
            mask = torch.zeros(d.pos.shape[0], dtype=torch.bool, device=d.device); mask[S] = True
            later = torch.cat(set_t[i_s + 1:]) if cfg['ignore_later'] and i_s + 1 < len(set_t) else None
            if later is not None:                     # §3.6: subsequent sets are ignored (their spots are free)
                fl0 = d.flags[later].clone()
                d.flags[later] = fl0 | ASIDE
                d.movable[later] = False
            if cfg['push_neighbors']:                 # variant: S first, then the non-critical cells it overlaps
                ill = ev.legality(per_cell=True)['illegal'] & d.movable & ~mask
                nb = torch.nonzero(ill).flatten()
                if len(nb):
                    fl1 = d.flags[nb].clone()
                    d.flags[nb] = fl1 | ASIDE; d.movable[nb] = False
                    naive.legalize(d, cells=mask, verbose=False)
                    d.flags[nb] = fl1; d.movable[nb] = True
                    m2 = torch.zeros_like(mask); m2[nb] = True
                    naive.legalize(d, cells=m2, verbose=False)
                    diag['pushed'] = diag.get('pushed', 0) + len(nb)
                else:
                    naive.legalize(d, cells=mask, verbose=False)
            else:
                naive.legalize(d, cells=mask, verbose=False)
            if later is not None:
                d.flags[later] = fl0
                d.movable[later] = True
            if cfg['diag']:
              with torch.no_grad():
                t_post = float(ds.forward(d.pos)[0])
              diag['pre'] += t_pre - t_b; diag['post'] += t_post - t_b
            diag['move_um'] += float((xy.float() - base[S]).abs().sum(1).mean()) / len(sets)
            diag['lg_um'] += float((d.pos[S] - xy.float()).abs().sum(1).mean()) / len(sets)
            base = d.pos.clone()
        res = exact(cfg['filter'])
        acc = res['tns'] > best['tns'] + 1e-6 or (res['tns'] >= best['tns'] - 1e-6 and res['wns'] > best['wns'])
        if cfg['filter'] == 'both':
            acc = acc and res['p_tns'] >= best['p_tns'] - 1e-6
        legal = bool(ev.legality()['legal'])
        if not legal:
            acc = False
        for c_, r_ in list(tabu_of.items()):
            if cfg['tabu_rounds'] > 0 and r - r_ >= cfg['tabu_rounds']:
                tabu.discard(c_); del tabu_of[c_]
        if acc:
            best = res; bad = 0; lr_scale = 1.0
            if cfg['tabu_rounds'] < 0:               # the placement changed: rejected cells may be tried again
                tabu.clear(); tabu_of.clear()
        else:
            d.pos = pos_r; d.orient = ori_r; d.update_orient()
            bad += 1
            lr_scale *= cfg['revert_lr']
            if cfg['tabu_rounds'] != 0:
                for c_ in crit.tolist():
                    tabu.add(c_); tabu_of[c_] = r
        h = {'round': r, 'tns': res['tns'], 'wns': res['wns'], 'hpwl': float(d.hpwl()), 'accepted': acc,
             'legal': legal, 'k': k, 'sets': len(sets), 'dsta_tns': tns_d, 't': time.time() - t_start,
             'round_s': time.time() - tr, 'diag': diag}
        hist.append(h)
        if verbose:
            print('round %2d  TNS %.1f WNS %.2f HPWL %.1f  %s  k=%d sets=%d  %.1fs | dsta gain pre-LG %.1f post-LG %.1f '
                  'move %.2fum lg %.2fum push %d' % (r, res['tns'], res['wns'], h['hpwl'], 'acc' if acc else 'REVERT', k,
                                             len(sets), h['round_s'], diag['pre'], diag['post'], diag['move_um'],
                                             diag['lg_um'], diag.get('pushed', 0)), flush=True)
        if bad >= cfg['patience']:
            break
    d.timing()
    if log_path:
        json.dump({'cfg': cfg, 'rounds': hist}, open(log_path, 'w'), indent=1)
    return hist

## 3. Design, ruler and starting placement

* **Design:** `after_fp/iccad2025/des` from Attar's zoo (downloaded on first use, about 10 s).
* **Ruler** (as the paper's "timing after early global route"): Attar's global route, then static timing with the
  routed parasitics. The contest SDC propagates an unbuffered clock net; before clock-tree synthesis the clock is
  timed as ideal, with the SDC's per-register clock latencies (useful skew), which is what OpenSTA does pre-CTS.
* **Start:** the paper starts from DREAMPlace. DREAMPlace is not installed here; its placement of `des` is embedded
  below. For another design, the start falls back to Attar's own global placer + legalizer.

In [ ]:
# DREAMPlace (global + legalization + detailed placement) of des, Bookshelf .pl, gzip + base64
START_PL = {'des': '...'}

In [ ]:
import base64, gzip, json, os, shutil, time
import numpy as np
import attar_patches  # noqa: F401  (the two fixes above)
from attar import naive, Evaluator

DESIGN = 'des'          # any of: ac97_top aes des des3 fpu mc_top pci_bridge32 tv80s (only des has the DREAMPlace start)

# pre-CTS copy of the design: the same data, the clock timed as ideal (with its pin latencies)
src = attar.load('after_fp/iccad2025/' + DESIGN, device='cpu').path
dst = os.path.join('designs_ideal', DESIGN)
shutil.copytree(src, dst, dirs_exist_ok=True)
tj = json.load(open(os.path.join(src, 'timing.json')))
for c in tj['clocks'].values():
    c['propagated'] = False
json.dump(tj, open(os.path.join(dst, 'timing.json'), 'w'))
d = attar.load(dst, device=DEVICE)
print('%s: %d cells, %d nets' % (DESIGN, len(d.names['cells']), len(d.n_ptr) - 1))


def set_start(d, design):
    if design in START_PL:
        idx = {n.replace('\\', ''): i for i, n in enumerate(d.names['cells'])}
        ii, xy = [], []
        for line in gzip.decompress(base64.b64decode(START_PL[design])).decode().splitlines():
            t = line.split()
            if len(t) >= 3 and not line.startswith(('UCLA', '#')) and t[0] in idx and bool(d.movable[idx[t[0]]]):
                ii.append(idx[t[0]]); xy.append((float(t[1]) / d.dbu, float(t[2]) / d.dbu))
        p = d.pos.clone()
        p[torch.as_tensor(ii, device=d.device)] = torch.as_tensor(xy, device=d.device, dtype=p.dtype)
        d.pos = p
        how = 'DREAMPlace (%d cells placed)' % len(ii)
    else:
        naive.global_place(d, verbose=False)
        how = 'Attar global placement'
    if not Evaluator(d).legality()['legal']:                # rows' rail orientation, sites: repair in place
        naive.legalize(d, verbose=False, incremental=True)
    assert Evaluator(d).legality()['legal']
    return how


def evaluate(d):
    # legality, HPWL; GR wirelength and timing with GR parasitics (the ruler)
    routes, st = d.global_route(verbose=False)
    g = d.timing(routes=routes).summary()
    out = {'legal': bool(Evaluator(d).legality()['legal']), 'hpwl': float(d.hpwl()),
           'wl': float(st['wirelength_um']), 'wns': g['wns'], 'tns': g['tns']}
    d.timing()
    return out


print('start:', set_start(d, DESIGN))
pos0, ori0 = d.pos.clone(), d.orient.clone()
e0 = evaluate(d)
print('start  GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s' % (e0['wl'], e0['wns'], e0['tns'], e0['legal']))

## 4. Run LATTE

Each round: exact timing at the current legal placement anchors the differentiable timer; the cells with the largest
timing gradient are split into independent sets; each set moves by Adam with CAWP preconditioning on
-TNS - λ1·WNS + λ2·WL and is legalized incrementally by Attar; the round is kept only if the exact (routed) TNS
improves. The configuration is the one used for the full reproduction (`ROUNDS` and the time limit are cut for Colab).

In [ ]:
import latte

ROUNDS = 12                 # the full reproduction runs up to 200 rounds (2 h limit); 12 give most of the gain on des
cfg = {'rounds': ROUNDS, 'patience': 40, 'lr': 0.5, 'k_frac': 0.01, 'lam1': 50, 'lam2': 0.1,
       'time_limit': 20 * 60}
t = time.time()
hist = latte.run(d, cfg)
rt = time.time() - t
e1 = evaluate(d)
pos_latte = d.pos.clone()
print('\nLATTE  GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s   (%.0f s)' % (e1['wl'], e1['wns'], e1['tns'],
                                                                              e1['legal'], rt))

## 5. Baseline: Attar's WL-driven detailed placer, from the same start

In [ ]:
d.pos, d.orient = pos0.clone(), ori0.clone(); d.update_orient()
t = time.time()
naive.detailed_place(d, verbose=False)
e2 = evaluate(d)
print('A-DP   GR WL %.0f um   WNS %.2f ps   TNS %.1f ps   legal %s   (%.0f s)' % (e2['wl'], e2['wns'], e2['tns'],
                                                                              e2['legal'], time.time() - t))
d.pos = pos_latte

import pandas as pd
rows = {'start (DREAMPlace)': e0, 'A-DP (Attar, WL-driven)': e2, 'LATTE, %d rounds' % (len(hist) - 1): e1}
df = pd.DataFrame({k: {'GR WL (um)': v['wl'], 'WNS (ps)': v['wns'], 'TNS (ps)': v['tns'],
                       'WL ratio': v['wl'] / e0['wl'], 'WNS ratio': v['wns'] / e0['wns'],
                       'TNS ratio': v['tns'] / e0['tns'], 'legal': v['legal']} for k, v in rows.items()}).T
ref = pd.DataFrame({'WL ratio': [1.007, 1.006, 1.041], 'WNS ratio': [0.938, 0.928, 0.884],
                    'TNS ratio': [0.886, 0.861, 0.868]},
                   index=['our full run on des (200 rounds)', 'our full run, 8-design geo-mean',
                          'paper, geo-mean'])
display(df.round(3)); display(ref)

## 6. Convergence and what moved

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
r = [h['round'] for h in hist]; tns = [h['tns'] for h in hist]
best = np.maximum.accumulate(tns)
a0 = ax[0]
a0.plot(r, best, color='#2a6fdb', lw=2)
acc = [h for h in hist[1:] if h['accepted']]; rej = [h for h in hist[1:] if not h['accepted']]
a0.scatter([h['round'] for h in acc], [h['tns'] for h in acc], s=36, color='#2a6fdb', zorder=3, label='accepted round')
a0.scatter([h['round'] for h in rej], [h['tns'] for h in rej], s=36, facecolor='white', edgecolor='#888',
           zorder=3, label='reverted round (TNS got worse)')
a0.axhline(e2['tns'], color='#c0504d', ls='--', lw=1)
a0.text(r[0], e2['tns'], 'A-DP (WL-driven DP)', color='#c0504d', va='bottom')
a0.set_xlabel('round'); a0.set_ylabel('TNS after global routing (ps)'); a0.set_title('LATTE on %s' % DESIGN)
a0.grid(alpha=.3); a0.legend(frameon=False, loc='lower right')

a1 = ax[1]
p0, p1 = pos0.cpu().numpy(), pos_latte.cpu().numpy()
mv = np.abs(p1 - p0).sum(1) > 1e-6
a1.scatter(p0[:, 0], p0[:, 1], s=1, color='#d0d0d0')
dx = p1[mv] - p0[mv]
a1.scatter(p0[mv, 0], p0[mv, 1], s=14, facecolor='white', edgecolor='#2a6fdb', zorder=3, label='before')
a1.scatter(p1[mv, 0], p1[mv, 1], s=14, color='#2a6fdb', zorder=4, label='after')
a1.quiver(p0[mv, 0], p0[mv, 1], dx[:, 0], dx[:, 1], angles='xy', scale_units='xy', scale=1, width=0.004,
          color='#2a6fdb', zorder=4)
a1.legend(frameon=False, loc='upper right')
a1.set_aspect('equal'); a1.set_title('%d cells moved, mean %.2f um (true scale)' % (mv.sum(), np.abs(dx).sum(1).mean()))
a1.set_xlabel('x (um)'); a1.set_ylabel('y (um)')
plt.tight_layout(); plt.show()

## Notes

* **Full reproduction** (8 ICCAD 2025 IWLS designs, DREAMPlace start, up to 2 h each): TNS geo-mean 0.861 vs the
  paper's 0.868 (OpenROAD's own global route + OpenSTA give 0.892), WL +0.6% vs the paper's +4.1%; LATTE beats
  WL-driven detailed placers, DREAMPlace 4.0 timing-driven placement and Rsyn (ICCAD 2015 incremental timing-driven
  placement). Those baselines need native tools (DREAMPlace, rsyn-x, OpenROAD) and are not run here.
* Cells move only a fraction of a micron: on these ASAP7 designs timing is mostly cell delay, so the paper's
  larger wirelength increase does not appear.
* Bigger designs: set `DESIGN` (they start from Attar's own placement) and use a GPU runtime; fpu and
  pci_bridge32 are several times larger than des and slower per round on CPU.